# 🩺 Sistema de apoyo al tamizaje de anemia por conjuntiva palpebral — Notebook V2 metodológicamente limpio

Este notebook reconstruye el experimento para tesis con un flujo defendible:

1. Auditar el dataset original exportado desde Roboflow.
2. Separar correctamente dos problemas distintos:
   - **YOLO**: detector anatómico de conjuntiva palpebral.
   - **Clasificador**: anemia vs normal usando únicamente el crop de conjuntiva.
3. Remapear las cajas YOLO originales a una sola clase anatómica: `conjuntiva`.
4. Conservar la etiqueta clínica `Anemia/Normal` en un CSV independiente.
5. Generar crops de conjuntiva desde las cajas validadas.
6. Entrenar y comparar varios clasificadores.
7. Calibrar temperatura y umbral **solo con validación**.
8. Evaluar el test set **una sola vez al final**.
9. Evaluar también el pipeline completo: imagen completa → YOLO → crop → clasificador.

> Nota metodológica: este sistema debe presentarse como **apoyo al tamizaje**, no como diagnóstico clínico definitivo, salvo que exista validación clínica con hemoglobina/hemograma y un protocolo de adquisición controlado.


## 0. Cambios críticos frente al notebook V1

El notebook anterior mezclaba varias decisiones que debilitan una tesis:

- El `data.yaml` original puede tener clases clínicas `Anemia` y `Normal`. Eso no debe ser usado como clases del detector anatómico final.
- El detector debe aprender `conjuntiva`, no `anemia`.
- El clasificador debe entrenarse con **crops de conjuntiva**, porque en inferencia recibirá crops.
- El umbral y la calibración se eligen con `val`, no con `test`.
- El test se usa al final y no se vuelve a tocar.
- Los labels malformados no se corrigen asignando anemia por defecto; se reportan y se excluyen o se corrigen manualmente.


## 1. Instalación opcional de dependencias

Ejecuta esta celda solo si tu entorno no tiene las librerías. En local/Colab puede tardar.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
import sys
import subprocess
from importlib.util import find_spec

REQUIRED_PACKAGES = {
    "ultralytics": "ultralytics",
    "torch": "torch",
    "torchvision": "torchvision",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "numpy": "numpy",
    "PIL": "pillow",
    "yaml": "pyyaml",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
}

INSTALL_MISSING_PACKAGES = False  # Cambia a True si estás en Colab o entorno limpio.

missing = []
for import_name, package_name in REQUIRED_PACKAGES.items():
    if find_spec(import_name) is None:
        missing.append(package_name)

print("Dependencias faltantes:", missing if missing else "ninguna")

if INSTALL_MISSING_PACKAGES and missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Instalación finalizada. Reinicia el kernel si alguna librería no carga.")


## 2. Configuración reproducible

Ajusta `PROJECT_ROOT` si el notebook no está en la raíz del proyecto. La raíz debe contener:

```text
project_root/
  data.yaml
  dataset/
    train/images
    train/labels
    val/images
    val/labels
    test/images
    test/labels
```


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
from pathlib import Path
import os
import re
import json
import math
import time
import shutil
import hashlib
import random
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import yaml

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    brier_score_loss,
)

SEED = 42

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

PROJECT_ROOT = Path.cwd()
RAW_DATA_YAML = PROJECT_ROOT / "data.yaml"
DATASET_ROOT = PROJECT_ROOT / "dataset"

WORK_DIR = PROJECT_ROOT / "tesis_outputs"
DETECTOR_DATASET = WORK_DIR / "dataset_yolo_conjuntiva"
CROPS_DATASET = WORK_DIR / "dataset_crops_conjuntiva"
REPORTS_DIR = WORK_DIR / "reports"
MODELS_DIR = WORK_DIR / "models"

for directory in [WORK_DIR, DETECTOR_DATASET, CROPS_DATASET, REPORTS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SPLITS = ("train", "val", "test")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Etiqueta clínica original del dataset Roboflow.
# IMPORTANTE: se usa para el clasificador, no para el detector anatómico.
ORIGINAL_CLASS_NAMES = {
    0: "Anemia",
    1: "Normal",
}
ANEMIA_NAME = "Anemia"
NORMAL_NAME = "Normal"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_YAML exists:", RAW_DATA_YAML.exists())
print("DATASET_ROOT exists:", DATASET_ROOT.exists())


## 3. Lectura del `data.yaml` original

El `data.yaml` original se usa para ubicar imágenes y para recuperar la etiqueta clínica. No se usará directamente para entrenar el detector final si sus clases son `Anemia/Normal`.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def read_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"No existe: {path}")
    with path.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f)

raw_cfg = read_yaml(RAW_DATA_YAML)
print(json.dumps(raw_cfg, indent=2, ensure_ascii=False))

names = raw_cfg.get("names")
nc = raw_cfg.get("nc")
print("\nDiagnóstico data.yaml:")
print("nc:", nc)
print("names:", names)

if nc == 2 and names and set(map(str.lower, names)) == {"anemia", "normal"}:
    print("⚠️ El data.yaml define clases clínicas. Se remapeará para YOLO anatómico: 0 = conjuntiva.")
elif nc == 1:
    print("✅ El data.yaml ya parece ser anatómico de una clase.")
else:
    print("⚠️ Revisar manualmente las clases del data.yaml.")


## 4. Auditoría fuerte del dataset

Objetivo: detectar errores antes de entrenar.

El auditor revisa:

- imágenes por split;
- labels por split;
- labels faltantes;
- imágenes sin label;
- labels sin imagen;
- clases encontradas;
- boxes fuera de rango;
- duplicados por nombre base antes de `.rf.`;
- posible fuga entre `train`, `val` y `test`.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def resolve_split_image_dir(cfg: dict, split: str, project_root: Path) -> Path:
    raw = cfg.get(split)
    if raw is None:
        raise KeyError(f"data.yaml no contiene split: {split}")
    p = Path(raw)
    if not p.is_absolute():
        p = project_root / p
    return p


def label_dir_from_image_dir(image_dir: Path) -> Path:
    if image_dir.name == "images":
        return image_dir.parent / "labels"
    return image_dir.parent / "labels"


def source_id_from_stem(stem: str) -> str:
    # Roboflow suele usar: nombre_original.rf.hash
    if ".rf." in stem:
        return stem.split(".rf.")[0]
    return stem


def parse_yolo_label(label_path: Path) -> List[dict]:
    rows = []
    if not label_path.exists():
        return rows
    text = label_path.read_text(encoding="utf-8", errors="ignore").strip()
    if not text:
        return rows
    for line_no, line in enumerate(text.splitlines(), start=1):
        parts = line.strip().split()
        if len(parts) != 5:
            rows.append({"valid": False, "line_no": line_no, "reason": "len != 5", "raw": line})
            continue
        try:
            class_id = int(float(parts[0]))
            xc, yc, w, h = map(float, parts[1:])
            in_range = (0 <= xc <= 1) and (0 <= yc <= 1) and (0 < w <= 1) and (0 < h <= 1)
            rows.append({
                "valid": bool(in_range),
                "line_no": line_no,
                "class_id": class_id,
                "xc": xc,
                "yc": yc,
                "w": w,
                "h": h,
                "reason": "ok" if in_range else "box_out_of_range",
                "raw": line,
            })
        except Exception as exc:
            rows.append({"valid": False, "line_no": line_no, "reason": f"parse_error: {exc}", "raw": line})
    return rows


def safe_image_size(path: Path) -> Tuple[Optional[int], Optional[int], Optional[str]]:
    try:
        with Image.open(path) as img:
            return img.width, img.height, None
    except Exception as exc:
        return None, None, str(exc)


def collect_dataset_index(cfg: dict, project_root: Path) -> pd.DataFrame:
    records = []
    for split in SPLITS:
        image_dir = resolve_split_image_dir(cfg, split, project_root)
        label_dir = label_dir_from_image_dir(image_dir)
        if not image_dir.exists():
            warnings.warn(f"No existe image_dir para {split}: {image_dir}")
            continue
        images = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS])
        label_files = sorted(label_dir.glob("*.txt")) if label_dir.exists() else []
        label_map = {p.stem: p for p in label_files}
        image_stems = {p.stem for p in images}

        # Imágenes con o sin label.
        for img_path in images:
            label_path = label_map.get(img_path.stem)
            parsed = parse_yolo_label(label_path) if label_path else []
            valid_rows = [r for r in parsed if r.get("valid")]
            class_ids = sorted(set([r["class_id"] for r in valid_rows]))
            width, height, image_error = safe_image_size(img_path)

            clinical_label_id = None
            clinical_label_name = None
            label_status = "ok"
            if image_error:
                label_status = "image_error"
            elif label_path is None:
                label_status = "missing_label"
            elif not parsed:
                label_status = "empty_label"
            elif not valid_rows:
                label_status = "invalid_label"
            elif len(class_ids) > 1:
                label_status = "mixed_classes_in_one_image"
                clinical_label_id = class_ids[0]
            else:
                clinical_label_id = class_ids[0]

            if clinical_label_id is not None:
                clinical_label_name = ORIGINAL_CLASS_NAMES.get(clinical_label_id, f"class_{clinical_label_id}")

            records.append({
                "split": split,
                "image_path": str(img_path),
                "label_path": str(label_path) if label_path else None,
                "stem": img_path.stem,
                "source_id": source_id_from_stem(img_path.stem),
                "image_width": width,
                "image_height": height,
                "image_error": image_error,
                "label_exists": label_path is not None,
                "label_status": label_status,
                "num_label_lines": len(parsed),
                "num_valid_boxes": len(valid_rows),
                "class_ids": ",".join(map(str, class_ids)),
                "clinical_label_id": clinical_label_id,
                "clinical_label_name": clinical_label_name,
            })

        # Labels sin imagen.
        orphan_labels = [p for p in label_files if p.stem not in image_stems]
        for label_path in orphan_labels:
            records.append({
                "split": split,
                "image_path": None,
                "label_path": str(label_path),
                "stem": label_path.stem,
                "source_id": source_id_from_stem(label_path.stem),
                "image_width": None,
                "image_height": None,
                "image_error": "label_without_image",
                "label_exists": True,
                "label_status": "label_without_image",
                "num_label_lines": len(parse_yolo_label(label_path)),
                "num_valid_boxes": None,
                "class_ids": None,
                "clinical_label_id": None,
                "clinical_label_name": None,
            })
    return pd.DataFrame(records)

index_df = collect_dataset_index(raw_cfg, PROJECT_ROOT)
index_path = REPORTS_DIR / "dataset_index_audit.csv"
index_df.to_csv(index_path, index=False, encoding="utf-8")
print("Índice guardado en:", index_path)
print(index_df.head())


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def summarize_audit(df: pd.DataFrame) -> None:
    if df.empty:
        print("No se encontraron registros. Revisa PROJECT_ROOT y data.yaml.")
        return

    print("=" * 80)
    print("RESUMEN POR SPLIT")
    print("=" * 80)
    print(df.groupby("split").agg(
        rows=("stem", "count"),
        images=("image_path", lambda s: s.notna().sum()),
        labels=("label_path", lambda s: s.notna().sum()),
        valid_images=("label_status", lambda s: (s == "ok").sum()),
    ))

    print("\n" + "=" * 80)
    print("ESTADOS DE LABEL")
    print("=" * 80)
    print(df["label_status"].value_counts(dropna=False))

    print("\n" + "=" * 80)
    print("DISTRIBUCIÓN CLÍNICA ORIGINAL")
    print("=" * 80)
    ok = df[df["label_status"] == "ok"].copy()
    print(pd.crosstab(ok["split"], ok["clinical_label_name"], margins=True))

    print("\n" + "=" * 80)
    print("POSIBLE FUGA POR source_id ENTRE SPLITS")
    print("=" * 80)
    source_split = ok.groupby("source_id")["split"].nunique()
    leaking_sources = source_split[source_split > 1].index.tolist()
    print("source_id presentes en más de un split:", len(leaking_sources))
    if leaking_sources:
        leak_df = ok[ok["source_id"].isin(leaking_sources)].sort_values(["source_id", "split", "stem"])
        leak_path = REPORTS_DIR / "potential_split_leakage_by_source_id.csv"
        leak_df.to_csv(leak_path, index=False, encoding="utf-8")
        print("⚠️ Posible fuga guardada en:", leak_path)
        display(leak_df[["source_id", "split", "stem", "clinical_label_name"]].head(30))
    else:
        print("✅ No se detectó fuga por source_id.")

    print("\n" + "=" * 80)
    print("TAMAÑOS DE IMAGEN")
    print("=" * 80)
    print(ok[["image_width", "image_height"]].describe())

summarize_audit(index_df)


## 5. Regla metodológica: nada de arreglos silenciosos

En este notebook:

- Si un label está vacío, no se convierte a `Anemia` por defecto.
- Si un archivo está corrupto, se reporta.
- Si una imagen aparece en más de un split por `source_id`, se reporta como posible fuga.
- Si hay fuga fuerte, lo correcto es reconstruir los splits por paciente o por imagen original antes de entrenar.


## 6. Construir dataset YOLO anatómico de una sola clase

El dataset original puede contener clases clínicas. Para el detector final se remapea todo a:

```yaml
nc: 1
names: ['conjuntiva']
```

Las coordenadas de las cajas se conservan. Solo cambia el `class_id` a `0`.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def remap_label_to_single_conjunctiva(src_label: Path, dst_label: Path) -> int:
    rows = parse_yolo_label(src_label)
    valid_rows = [r for r in rows if r.get("valid")]
    dst_label.parent.mkdir(parents=True, exist_ok=True)
    with dst_label.open("w", encoding="utf-8") as f:
        for r in valid_rows:
            f.write(f"0 {r['xc']:.10f} {r['yc']:.10f} {r['w']:.10f} {r['h']:.10f}\n")
    return len(valid_rows)


def build_single_class_yolo_dataset(df: pd.DataFrame, output_root: Path) -> Path:
    ok = df[df["label_status"] == "ok"].copy()
    if ok.empty:
        raise RuntimeError("No hay imágenes válidas para crear dataset YOLO anatómico.")

    if output_root.exists():
        shutil.rmtree(output_root)
    for split in SPLITS:
        (output_root / split / "images").mkdir(parents=True, exist_ok=True)
        (output_root / split / "labels").mkdir(parents=True, exist_ok=True)

    copied = []
    for row in ok.itertuples(index=False):
        img_src = Path(row.image_path)
        lbl_src = Path(row.label_path)
        img_dst = output_root / row.split / "images" / img_src.name
        lbl_dst = output_root / row.split / "labels" / f"{img_src.stem}.txt"
        shutil.copy2(img_src, img_dst)
        n_boxes = remap_label_to_single_conjunctiva(lbl_src, lbl_dst)
        copied.append({"split": row.split, "image": img_dst.name, "boxes": n_boxes})

    yolo_yaml = output_root / "data_conjuntiva.yaml"
    yaml_payload = {
        "path": str(output_root.resolve()),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "nc": 1,
        "names": ["conjuntiva"],
    }
    yolo_yaml.write_text(yaml.safe_dump(yaml_payload, sort_keys=False, allow_unicode=True), encoding="utf-8")

    copied_df = pd.DataFrame(copied)
    copied_df.to_csv(REPORTS_DIR / "single_class_yolo_dataset_manifest.csv", index=False, encoding="utf-8")
    print("Dataset YOLO anatómico creado en:", output_root)
    print("YAML:", yolo_yaml)
    print(copied_df.groupby("split").agg(images=("image", "count"), boxes=("boxes", "sum")))
    return yolo_yaml

CONJUNCTIVA_YAML = build_single_class_yolo_dataset(index_df, DETECTOR_DATASET)
print(CONJUNCTIVA_YAML.read_text(encoding="utf-8"))


## 7. Entrenar YOLOv8 como detector de conjuntiva

Este entrenamiento ya no usa `Anemia/Normal` como clases del detector. Solo detecta la región anatómica.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
RUN_YOLO_TRAINING = False  # Cambia a True cuando la auditoría esté limpia.
YOLO_BASE_MODEL = "yolov8n.pt"
YOLO_RUN_NAME = "conjuntiva_detector_v2_single_class"
YOLO_EPOCHS = 50
YOLO_IMGSZ = 640
YOLO_BATCH = 16

if RUN_YOLO_TRAINING:
    from ultralytics import YOLO
    yolo_model = YOLO(YOLO_BASE_MODEL)
    yolo_results = yolo_model.train(
        data=str(CONJUNCTIVA_YAML),
        epochs=YOLO_EPOCHS,
        imgsz=YOLO_IMGSZ,
        batch=YOLO_BATCH,
        patience=10,
        project=str(WORK_DIR / "runs_detect"),
        name=YOLO_RUN_NAME,
        exist_ok=True,
        seed=SEED,
    )
    print("Entrenamiento YOLO finalizado:", yolo_results)
else:
    print("RUN_YOLO_TRAINING=False. Actívalo solo después de revisar la auditoría.")


## 8. Generar crops de conjuntiva desde labels validados

Para entrenar el clasificador, se usan crops de la conjuntiva. Eso alinea entrenamiento e inferencia.

Decisión metodológica:

- Para entrenar el clasificador se usan cajas ground truth del dataset.
- Para evaluación del pipeline completo se usa YOLO predicho sobre la imagen completa.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def yolo_box_to_xyxy(xc: float, yc: float, w: float, h: float, img_w: int, img_h: int, margin: float = 0.08) -> Tuple[int, int, int, int]:
    x1 = (xc - w / 2) * img_w
    y1 = (yc - h / 2) * img_h
    x2 = (xc + w / 2) * img_w
    y2 = (yc + h / 2) * img_h

    bw = x2 - x1
    bh = y2 - y1
    x1 -= bw * margin
    y1 -= bh * margin
    x2 += bw * margin
    y2 += bh * margin

    x1 = max(0, int(round(x1)))
    y1 = max(0, int(round(y1)))
    x2 = min(img_w, int(round(x2)))
    y2 = min(img_h, int(round(y2)))
    return x1, y1, x2, y2


def choose_largest_valid_box(label_path: Path) -> Optional[dict]:
    rows = [r for r in parse_yolo_label(label_path) if r.get("valid")]
    if not rows:
        return None
    return max(rows, key=lambda r: r["w"] * r["h"])


def build_crop_dataset(df: pd.DataFrame, output_root: Path, margin: float = 0.08) -> pd.DataFrame:
    ok = df[df["label_status"] == "ok"].copy()
    if output_root.exists():
        shutil.rmtree(output_root)

    for split in SPLITS:
        for class_name in [ANEMIA_NAME, NORMAL_NAME]:
            (output_root / split / class_name).mkdir(parents=True, exist_ok=True)

    crop_records = []
    errors = []
    for row in ok.itertuples(index=False):
        img_path = Path(row.image_path)
        label_path = Path(row.label_path)
        clinical_label = row.clinical_label_name
        if clinical_label not in {ANEMIA_NAME, NORMAL_NAME}:
            errors.append({"image_path": str(img_path), "reason": f"clinical_label_invalid: {clinical_label}"})
            continue
        box = choose_largest_valid_box(label_path)
        if box is None:
            errors.append({"image_path": str(img_path), "reason": "no_valid_box"})
            continue
        try:
            with Image.open(img_path) as img:
                img = ImageOps.exif_transpose(img).convert("RGB")
                x1, y1, x2, y2 = yolo_box_to_xyxy(box["xc"], box["yc"], box["w"], box["h"], img.width, img.height, margin=margin)
                if x2 <= x1 or y2 <= y1:
                    errors.append({"image_path": str(img_path), "reason": "invalid_crop_dimensions"})
                    continue
                crop = img.crop((x1, y1, x2, y2))
                dst = output_root / row.split / clinical_label / f"{img_path.stem}_crop.jpg"
                crop.save(dst, quality=95)
                crop_records.append({
                    "split": row.split,
                    "source_image": str(img_path),
                    "crop_path": str(dst),
                    "clinical_label": clinical_label,
                    "clinical_label_id": int(row.clinical_label_id),
                    "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                    "crop_width": x2 - x1,
                    "crop_height": y2 - y1,
                    "source_id": row.source_id,
                })
        except Exception as exc:
            errors.append({"image_path": str(img_path), "reason": str(exc)})

    crop_df = pd.DataFrame(crop_records)
    error_df = pd.DataFrame(errors)
    crop_df.to_csv(REPORTS_DIR / "crop_dataset_manifest.csv", index=False, encoding="utf-8")
    error_df.to_csv(REPORTS_DIR / "crop_dataset_errors.csv", index=False, encoding="utf-8")

    print("Crops creados en:", output_root)
    print(pd.crosstab(crop_df["split"], crop_df["clinical_label"], margins=True))
    print("Errores:", len(error_df))
    if len(error_df):
        display(error_df.head())
    return crop_df

crop_df = build_crop_dataset(index_df, CROPS_DATASET, margin=0.08)


## 9. Visualizar muestras de crops

Antes de entrenar, revisa visualmente si los crops realmente capturan conjuntiva inferior. Si los crops incluyen demasiada piel, iris o párpado, el clasificador puede aprender atajos.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def show_crop_samples(crop_manifest: pd.DataFrame, n_per_class: int = 6) -> None:
    if crop_manifest.empty:
        print("No hay crops para visualizar.")
        return
    sample_rows = []
    for cls in [ANEMIA_NAME, NORMAL_NAME]:
        sub = crop_manifest[crop_manifest["clinical_label"] == cls]
        if len(sub):
            sample_rows.append(sub.sample(min(n_per_class, len(sub)), random_state=SEED))
    sample_df = pd.concat(sample_rows, ignore_index=True) if sample_rows else pd.DataFrame()
    if sample_df.empty:
        print("Sin muestras.")
        return

    cols = n_per_class
    rows = len(sample_df["clinical_label"].unique())
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.2))
    axes = np.array(axes).reshape(rows, cols)
    for r, cls in enumerate([ANEMIA_NAME, NORMAL_NAME]):
        sub = sample_df[sample_df["clinical_label"] == cls].reset_index(drop=True)
        for c in range(cols):
            ax = axes[r, c]
            ax.axis("off")
            if c < len(sub):
                img = Image.open(sub.loc[c, "crop_path"]).convert("RGB")
                ax.imshow(img)
                ax.set_title(cls, fontsize=9)
    plt.tight_layout()
    out = REPORTS_DIR / "crop_samples.png"
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.show()
    print("Figura guardada en:", out)

show_crop_samples(crop_df, n_per_class=6)


## 10. DataLoaders para clasificación con crops

Se evita alterar demasiado el color porque el color de la conjuntiva puede ser una señal clínica. La aumentación es moderada.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2 if os.name != "nt" else 0

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=7),
    transforms.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.05, hue=0.015),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def make_imagefolder(split: str, transform):
    path = CROPS_DATASET / split
    if not path.exists():
        raise FileNotFoundError(f"No existe: {path}")
    return datasets.ImageFolder(root=str(path), transform=transform)

train_dataset = make_imagefolder("train", train_transforms)
val_dataset = make_imagefolder("val", val_transforms)
test_dataset = make_imagefolder("test", val_transforms)

print("class_to_idx:", train_dataset.class_to_idx)
assert ANEMIA_NAME in train_dataset.class_to_idx, "Falta clase Anemia en crops/train"
assert NORMAL_NAME in train_dataset.class_to_idx, "Falta clase Normal en crops/train"

ANEMIA_IDX = train_dataset.class_to_idx[ANEMIA_NAME]
NORMAL_IDX = train_dataset.class_to_idx[NORMAL_NAME]
IDX_TO_CLASS = {v: k for k, v in train_dataset.class_to_idx.items()}


def make_weighted_sampler(dataset: datasets.ImageFolder) -> WeightedRandomSampler:
    labels = np.array([y for _, y in dataset.samples])
    class_counts = np.bincount(labels, minlength=len(dataset.classes))
    class_counts = np.maximum(class_counts, 1)
    weights = np.array([1.0 / class_counts[y] for y in labels], dtype=np.float64)
    return WeightedRandomSampler(weights=torch.DoubleTensor(weights), num_samples=len(weights), replacement=True)

train_sampler = make_weighted_sampler(train_dataset)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

print("Train:", len(train_dataset), "Val:", len(val_dataset), "Test:", len(test_dataset))
print("Train classes:", np.bincount([y for _, y in train_dataset.samples]))


## 11. Fábrica de modelos comparativos

Para tesis no basta usar un solo modelo. Compara al menos:

- EfficientNet-B0
- ResNet18
- MobileNetV3-Small

La selección se hace por `F1 macro` en validación, no por test.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def build_model(model_name: str, num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    model_name = model_name.lower()

    if model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
        return model

    if model_name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)
        return model

    if model_name == "mobilenet_v3_small":
        weights = models.MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        model = models.mobilenet_v3_small(weights=weights)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model

    raise ValueError(f"Modelo no soportado: {model_name}")


def class_weights_from_dataset(dataset: datasets.ImageFolder) -> torch.Tensor:
    labels = np.array([y for _, y in dataset.samples])
    counts = np.bincount(labels, minlength=len(dataset.classes)).astype(np.float32)
    counts = np.maximum(counts, 1.0)
    weights = len(labels) / (len(dataset.classes) * counts)
    return torch.tensor(weights, dtype=torch.float32)

CLASS_WEIGHTS = class_weights_from_dataset(train_dataset).to(DEVICE)
print("CLASS_WEIGHTS:", CLASS_WEIGHTS)


## 12. Entrenamiento y evaluación en validación

El mejor checkpoint se elige por `F1 macro` en validación.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
@dataclass
class TrainConfig:
    epochs: int = 20
    lr: float = 1e-4
    weight_decay: float = 1e-4
    patience: int = 5
    pretrained: bool = True


def run_epoch_train(model: nn.Module, loader: DataLoader, optimizer, criterion) -> float:
    model.train()
    total_loss = 0.0
    total_n = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * images.size(0)
        total_n += images.size(0)
    return total_loss / max(total_n, 1)


@torch.no_grad()
def collect_logits(model: nn.Module, loader: DataLoader) -> Tuple[np.ndarray, np.ndarray, float]:
    model.eval()
    logits_list = []
    labels_list = []
    total_loss = 0.0
    total_n = 0
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        logits_list.append(logits.detach().cpu().numpy())
        labels_list.append(labels.detach().cpu().numpy())
        total_loss += float(loss.item()) * images.size(0)
        total_n += images.size(0)
    return np.concatenate(logits_list), np.concatenate(labels_list), total_loss / max(total_n, 1)


def metrics_from_logits(y_true: np.ndarray, logits: np.ndarray) -> dict:
    proba = torch.softmax(torch.tensor(logits), dim=1).numpy()
    y_pred = proba.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }


def train_classifier(model_name: str, cfg: TrainConfig) -> dict:
    set_seed(SEED)
    model = build_model(model_name, num_classes=2, pretrained=cfg.pretrained).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_f1 = -1.0
    best_path = MODELS_DIR / f"best_{model_name}.pth"
    history = []
    bad_epochs = 0

    for epoch in range(1, cfg.epochs + 1):
        train_loss = run_epoch_train(model, train_loader, optimizer, criterion)
        val_logits, val_labels, val_loss = collect_logits(model, val_loader)
        val_metrics = metrics_from_logits(val_labels, val_logits)
        scheduler.step(val_metrics["f1_macro"])

        row = {
            "model_name": model_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            **val_metrics,
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        print(row)

        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            bad_epochs = 0
            torch.save({
                "model_name": model_name,
                "state_dict": model.state_dict(),
                "class_to_idx": train_dataset.class_to_idx,
                "idx_to_class": IDX_TO_CLASS,
                "img_size": IMG_SIZE,
                "seed": SEED,
                "best_val_f1_macro": best_f1,
            }, best_path)
        else:
            bad_epochs += 1
            if bad_epochs >= cfg.patience:
                print(f"Early stopping en epoch {epoch}")
                break

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(REPORTS_DIR / f"history_{model_name}.csv", index=False, encoding="utf-8")
    return {"model_name": model_name, "best_path": str(best_path), "best_val_f1_macro": best_f1, "history": hist_df}


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
RUN_CLASSIFIER_TRAINING = False  # Cambia a True para entrenar.
MODEL_CANDIDATES = ["efficientnet_b0", "resnet18", "mobilenet_v3_small"]
TRAIN_CFG = TrainConfig(epochs=20, lr=1e-4, weight_decay=1e-4, patience=5, pretrained=True)

training_results = []
if RUN_CLASSIFIER_TRAINING:
    for model_name in MODEL_CANDIDATES:
        print("\n" + "=" * 90)
        print("ENTRENANDO:", model_name)
        print("=" * 90)
        result = train_classifier(model_name, TRAIN_CFG)
        training_results.append(result)

    results_df = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in training_results])
    results_df = results_df.sort_values("best_val_f1_macro", ascending=False)
    results_df.to_csv(REPORTS_DIR / "model_comparison_val.csv", index=False, encoding="utf-8")
    display(results_df)
else:
    print("RUN_CLASSIFIER_TRAINING=False.")
    print("Cuando entrenes, activa esta bandera y luego usa el mejor checkpoint de model_comparison_val.csv.")


## 13. Cargar mejor modelo entrenado

Después de entrenar, usa el checkpoint ganador por validación.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
BEST_MODEL_NAME = "efficientnet_b0"  # Cambia según model_comparison_val.csv.
BEST_MODEL_PATH = MODELS_DIR / f"best_{BEST_MODEL_NAME}.pth"


def load_classifier_checkpoint(path: Path, model_name: str) -> nn.Module:
    if not path.exists():
        raise FileNotFoundError(f"No existe checkpoint: {path}. Primero entrena o copia el modelo.")
    ckpt = torch.load(path, map_location=DEVICE)
    model = build_model(model_name, num_classes=2, pretrained=False).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model

if BEST_MODEL_PATH.exists():
    best_model = load_classifier_checkpoint(BEST_MODEL_PATH, BEST_MODEL_NAME)
    print("Modelo cargado:", BEST_MODEL_PATH)
else:
    best_model = None
    print("Aún no existe:", BEST_MODEL_PATH)


## 14. Temperature Scaling usando solo validación

La calibración se ajusta únicamente con logits de `val`.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
class TemperatureScaler(nn.Module):
    def __init__(self, init_temp: float = 1.5):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.log(torch.tensor([init_temp], dtype=torch.float32)))

    @property
    def temperature(self) -> torch.Tensor:
        return torch.exp(self.log_temperature).clamp(min=1e-3, max=100.0)

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.temperature

    def fit(self, logits_np: np.ndarray, labels_np: np.ndarray, max_iter: int = 100) -> float:
        logits = torch.tensor(logits_np, dtype=torch.float32, device=DEVICE)
        labels = torch.tensor(labels_np, dtype=torch.long, device=DEVICE)
        self.to(DEVICE)
        optimizer = torch.optim.LBFGS([self.log_temperature], lr=0.05, max_iter=max_iter)
        criterion = nn.CrossEntropyLoss()

        def closure():
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(self.forward(logits), labels)
            loss.backward()
            return loss

        optimizer.step(closure)
        return float(self.temperature.detach().cpu().item())


def apply_temperature_np(logits_np: np.ndarray, temperature: float) -> np.ndarray:
    return logits_np / float(temperature)

if best_model is not None:
    val_logits_raw, val_labels, val_loss = collect_logits(best_model, val_loader)
    scaler = TemperatureScaler(init_temp=1.5)
    temperature = scaler.fit(val_logits_raw, val_labels)
    val_logits_cal = apply_temperature_np(val_logits_raw, temperature)
    print("Temperatura calibrada con VAL:", temperature)
    print("Val metrics raw:", metrics_from_logits(val_labels, val_logits_raw))
    print("Val metrics calibrated:", metrics_from_logits(val_labels, val_logits_cal))
else:
    temperature = 1.0
    val_logits_cal = None
    val_labels = None
    print("Sin modelo cargado; calibración omitida.")


## 15. Selección de umbral usando solo validación

Se escoge el umbral para probabilidad de anemia con `val`, no con `test`.

El test no participa en esta decisión.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def softmax_np(logits_np: np.ndarray) -> np.ndarray:
    x = logits_np - logits_np.max(axis=1, keepdims=True)
    ex = np.exp(x)
    return ex / ex.sum(axis=1, keepdims=True)


def choose_threshold_from_val(y_true: np.ndarray, prob_anemia: np.ndarray, min_sensitivity: Optional[float] = None) -> dict:
    y_bin = (y_true == ANEMIA_IDX).astype(int)
    fpr, tpr, thresholds = roc_curve(y_bin, prob_anemia)
    specificity = 1.0 - fpr

    if min_sensitivity is not None:
        valid = np.where(tpr >= min_sensitivity)[0]
        if len(valid) > 0:
            # Entre los que cumplen sensibilidad mínima, elegimos mayor especificidad.
            idx = valid[np.argmax(specificity[valid])]
        else:
            idx = int(np.argmax(tpr - fpr))
    else:
        idx = int(np.argmax(tpr - fpr))  # Youden J

    threshold = float(thresholds[idx])
    if not np.isfinite(threshold):
        threshold = 0.5

    return {
        "threshold": threshold,
        "val_sensitivity_anemia": float(tpr[idx]),
        "val_specificity_normal": float(specificity[idx]),
        "youden_j": float(tpr[idx] - fpr[idx]),
        "min_sensitivity_requested": min_sensitivity,
    }

if val_logits_cal is not None:
    val_proba_cal = softmax_np(val_logits_cal)
    val_prob_anemia = val_proba_cal[:, ANEMIA_IDX]
    threshold_info = choose_threshold_from_val(val_labels, val_prob_anemia, min_sensitivity=None)
    THRESHOLD_ANEMIA = threshold_info["threshold"]
    print(json.dumps(threshold_info, indent=2))
else:
    THRESHOLD_ANEMIA = 0.5
    threshold_info = {"threshold": 0.5, "note": "default because model not loaded"}
    print(threshold_info)


## 16. Métricas finales con intervalo de confianza

El test se evalúa al final con:

- Accuracy
- Sensibilidad para anemia
- Especificidad para normal
- Precision
- F1 macro
- ROC-AUC
- PR-AUC
- Brier Score
- Matriz de confusión
- Intervalos de confianza Wilson 95% para métricas proporcionales


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def wilson_ci(successes: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    if n <= 0:
        return (float("nan"), float("nan"))
    phat = successes / n
    denom = 1 + z**2 / n
    center = (phat + z**2 / (2*n)) / denom
    margin = z * math.sqrt((phat*(1-phat) + z**2/(4*n)) / n) / denom
    return max(0.0, center - margin), min(1.0, center + margin)


def evaluate_binary_classifier(y_true: np.ndarray, prob_anemia: np.ndarray, threshold: float, title: str) -> dict:
    y_true_bin = (y_true == ANEMIA_IDX).astype(int)  # 1 = Anemia, 0 = Normal
    y_pred_bin = (prob_anemia >= threshold).astype(int)

    cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[1, 0])
    # filas: real anemia, real normal. cols: pred anemia, pred normal.
    tp = int(cm[0, 0])
    fn = int(cm[0, 1])
    fp = int(cm[1, 0])
    tn = int(cm[1, 1])

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    precision_anemia = tp / max(tp + fp, 1)
    f1_anemia = 2 * precision_anemia * sensitivity / max(precision_anemia + sensitivity, 1e-12)

    try:
        roc_auc = roc_auc_score(y_true_bin, prob_anemia)
    except Exception:
        roc_auc = float("nan")
    try:
        pr_auc = average_precision_score(y_true_bin, prob_anemia)
    except Exception:
        pr_auc = float("nan")
    try:
        brier = brier_score_loss(y_true_bin, prob_anemia)
    except Exception:
        brier = float("nan")

    metrics = {
        "title": title,
        "threshold_anemia": threshold,
        "n": int(len(y_true)),
        "tp_anemia": tp,
        "fn_anemia": fn,
        "fp_anemia": fp,
        "tn_normal": tn,
        "accuracy": accuracy,
        "accuracy_ci95_low": wilson_ci(tp + tn, tp + tn + fp + fn)[0],
        "accuracy_ci95_high": wilson_ci(tp + tn, tp + tn + fp + fn)[1],
        "sensitivity_anemia": sensitivity,
        "sensitivity_ci95_low": wilson_ci(tp, tp + fn)[0],
        "sensitivity_ci95_high": wilson_ci(tp, tp + fn)[1],
        "specificity_normal": specificity,
        "specificity_ci95_low": wilson_ci(tn, tn + fp)[0],
        "specificity_ci95_high": wilson_ci(tn, tn + fp)[1],
        "precision_anemia": precision_anemia,
        "f1_anemia": f1_anemia,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "brier_score": brier,
    }
    return metrics


def plot_confusion_binary(metrics: dict, out_path: Path) -> None:
    mat = np.array([[metrics["tp_anemia"], metrics["fn_anemia"]], [metrics["fp_anemia"], metrics["tn_normal"]]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(mat)
    ax.set_xticks([0, 1], labels=["Pred Anemia", "Pred Normal"])
    ax.set_yticks([0, 1], labels=["Real Anemia", "Real Normal"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(mat[i, j]), ha="center", va="center", fontsize=14)
    ax.set_title(metrics["title"])
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()


def plot_roc_pr(y_true: np.ndarray, prob_anemia: np.ndarray, out_path: Path) -> None:
    y_bin = (y_true == ANEMIA_IDX).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, prob_anemia)
    precision, recall, _ = precision_recall_curve(y_bin, prob_anemia)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label="ROC")
    ax.plot([0, 1], [0, 1], linestyle="--", label="Azar")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Curva ROC")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path.with_name(out_path.stem + "_roc.png"), dpi=180, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, label="PR")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title("Curva Precision-Recall")
    ax.legend()
    plt.tight_layout()
    plt.savefig(out_path.with_name(out_path.stem + "_pr.png"), dpi=180, bbox_inches="tight")
    plt.show()


## 17. Evaluación final del clasificador sobre crops de test

Esta es la primera evaluación sobre `test`. No se usa para recalibrar ni ajustar umbrales.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
if best_model is not None:
    test_logits_raw, test_labels, test_loss = collect_logits(best_model, test_loader)
    test_logits_cal = apply_temperature_np(test_logits_raw, temperature)
    test_proba_cal = softmax_np(test_logits_cal)
    test_prob_anemia = test_proba_cal[:, ANEMIA_IDX]

    crop_test_metrics = evaluate_binary_classifier(
        y_true=test_labels,
        prob_anemia=test_prob_anemia,
        threshold=THRESHOLD_ANEMIA,
        title="Clasificador sobre crops de conjuntiva - TEST",
    )
    pd.DataFrame([crop_test_metrics]).to_csv(REPORTS_DIR / "test_metrics_classifier_crops.csv", index=False, encoding="utf-8")
    print(json.dumps(crop_test_metrics, indent=2))
    plot_confusion_binary(crop_test_metrics, REPORTS_DIR / "test_confusion_classifier_crops.png")
    plot_roc_pr(test_labels, test_prob_anemia, REPORTS_DIR / "test_classifier_curves.png")
else:
    print("No hay modelo cargado. Entrena/carga el clasificador para evaluar test.")


## 18. Evaluación del pipeline completo: imagen completa → YOLO → crop → clasificador

Esta es la métrica más importante para la tesis, porque mide el sistema real.

Requisitos:

- YOLO entrenado como `conjuntiva`.
- Clasificador cargado.
- Temperatura y umbral ya definidos con `val`.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
YOLO_BEST_PATH = WORK_DIR / "runs_detect" / YOLO_RUN_NAME / "weights" / "best.pt"
YOLO_CONF = 0.25


def crop_from_yolo_prediction(image_path: Path, yolo_model, conf: float = 0.25, margin: float = 0.08) -> Tuple[Optional[Image.Image], dict]:
    result = yolo_model.predict(str(image_path), conf=conf, verbose=False)[0]
    if result.boxes is None or len(result.boxes) == 0:
        return None, {"detected": False, "reason": "no_detection"}
    boxes_xyxy = result.boxes.xyxy.detach().cpu().numpy()
    confs = result.boxes.conf.detach().cpu().numpy()
    best_idx = int(np.argmax(confs))
    x1, y1, x2, y2 = boxes_xyxy[best_idx].tolist()

    with Image.open(image_path) as img:
        img = ImageOps.exif_transpose(img).convert("RGB")
        bw = x2 - x1
        bh = y2 - y1
        x1 = max(0, int(round(x1 - bw * margin)))
        y1 = max(0, int(round(y1 - bh * margin)))
        x2 = min(img.width, int(round(x2 + bw * margin)))
        y2 = min(img.height, int(round(y2 + bh * margin)))
        if x2 <= x1 or y2 <= y1:
            return None, {"detected": False, "reason": "invalid_predicted_box"}
        crop = img.crop((x1, y1, x2, y2))
        return crop, {"detected": True, "conf": float(confs[best_idx]), "x1": x1, "y1": y1, "x2": x2, "y2": y2}


def predict_crop_with_classifier(crop: Image.Image, model: nn.Module, temperature_value: float) -> float:
    tensor = val_transforms(crop).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(tensor)
        logits = logits / float(temperature_value)
        prob = F.softmax(logits, dim=1)[0, ANEMIA_IDX].item()
    return float(prob)


def evaluate_full_pipeline(index: pd.DataFrame, yolo_path: Path, model: nn.Module) -> pd.DataFrame:
    from ultralytics import YOLO
    if not yolo_path.exists():
        raise FileNotFoundError(f"No existe YOLO entrenado: {yolo_path}")
    yolo_model = YOLO(str(yolo_path))
    test_df = index[(index["split"] == "test") & (index["label_status"] == "ok")].copy()

    records = []
    for row in test_df.itertuples(index=False):
        image_path = Path(row.image_path)
        crop, det_info = crop_from_yolo_prediction(image_path, yolo_model, conf=YOLO_CONF, margin=0.08)
        if crop is None:
            prob_anemia = float("nan")
            pred_label = "No detectado"
        else:
            prob_anemia = predict_crop_with_classifier(crop, model, temperature)
            pred_label = ANEMIA_NAME if prob_anemia >= THRESHOLD_ANEMIA else NORMAL_NAME
        records.append({
            "image_path": str(image_path),
            "true_label": row.clinical_label_name,
            "true_label_idx": int(row.clinical_label_id),
            "prob_anemia": prob_anemia,
            "pred_label": pred_label,
            **det_info,
        })
    return pd.DataFrame(records)

RUN_FULL_PIPELINE_EVAL = False

if RUN_FULL_PIPELINE_EVAL:
    if best_model is None:
        raise RuntimeError("Carga primero el clasificador.")
    full_df = evaluate_full_pipeline(index_df, YOLO_BEST_PATH, best_model)
    full_df.to_csv(REPORTS_DIR / "test_full_pipeline_predictions.csv", index=False, encoding="utf-8")
    display(full_df.head())

    detected_df = full_df[full_df["detected"] == True].copy()
    y_true_full = detected_df["true_label"].map({ANEMIA_NAME: ANEMIA_IDX, NORMAL_NAME: NORMAL_IDX}).to_numpy()
    prob_full = detected_df["prob_anemia"].to_numpy(dtype=float)
    full_metrics = evaluate_binary_classifier(y_true_full, prob_full, THRESHOLD_ANEMIA, "Pipeline completo - TEST")

    # Penalización adicional: reportar tasa de no detección.
    full_metrics["n_total_test"] = int(len(full_df))
    full_metrics["n_no_detection"] = int((full_df["detected"] != True).sum())
    full_metrics["detection_rate"] = float((full_df["detected"] == True).mean())

    pd.DataFrame([full_metrics]).to_csv(REPORTS_DIR / "test_metrics_full_pipeline.csv", index=False, encoding="utf-8")
    print(json.dumps(full_metrics, indent=2))
    plot_confusion_binary(full_metrics, REPORTS_DIR / "test_confusion_full_pipeline.png")
else:
    print("RUN_FULL_PIPELINE_EVAL=False. Actívalo cuando tengas YOLO y clasificador entrenados.")
    print("YOLO esperado en:", YOLO_BEST_PATH)


## 19. Sistema de calidad de imagen no bloqueante

Sirve para advertir baja calidad de imagen. No debe usarse para alterar resultados del test después de verlos.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def image_quality_report(image: Image.Image) -> dict:
    import cv2
    arr = np.array(image.convert("RGB"))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    blur_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    mean_intensity = float(gray.mean())
    contrast = float(gray.std())
    return {
        "blur_laplacian_var": blur_var,
        "mean_intensity": mean_intensity,
        "contrast_std": contrast,
        "is_blurry_warning": blur_var < 40.0,
        "is_dark_warning": mean_intensity < 50.0,
        "is_overexposed_warning": mean_intensity > 220.0,
        "is_low_contrast_warning": contrast < 20.0,
    }

# Ejemplo de uso:
# img = Image.open("ruta/a/imagen.jpg").convert("RGB")
# image_quality_report(img)
print("Función image_quality_report lista.")


## 20. Inferencia individual controlada

Esta función sirve para demo, después de cerrar evaluación. No debe usarse para ajustar el umbral mirando imágenes de test.


In [ ]:
# notebooks/cp_v2_tesis_pipeline_limpio.ipynb
def predict_single_image(image_path: str, yolo_path: Optional[str] = None) -> dict:
    if best_model is None:
        raise RuntimeError("No hay clasificador cargado.")
    img_path = Path(image_path)
    if not img_path.exists():
        raise FileNotFoundError(img_path)

    if yolo_path is None:
        # Modo crop manual: clasifica imagen completa o crop ya recortado.
        with Image.open(img_path) as img:
            img = ImageOps.exif_transpose(img).convert("RGB")
            quality = image_quality_report(img)
            prob_anemia = predict_crop_with_classifier(img, best_model, temperature)
            pred_label = ANEMIA_NAME if prob_anemia >= THRESHOLD_ANEMIA else NORMAL_NAME
            return {
                "mode": "classifier_only_assumes_input_is_crop",
                "image_path": str(img_path),
                "prob_anemia": prob_anemia,
                "threshold_anemia": THRESHOLD_ANEMIA,
                "prediction": pred_label,
                "quality": quality,
            }

    from ultralytics import YOLO
    yolo_model = YOLO(str(yolo_path))
    crop, det_info = crop_from_yolo_prediction(img_path, yolo_model, conf=YOLO_CONF, margin=0.08)
    if crop is None:
        return {"image_path": str(img_path), "prediction": "No detectado", **det_info}
    quality = image_quality_report(crop)
    prob_anemia = predict_crop_with_classifier(crop, best_model, temperature)
    pred_label = ANEMIA_NAME if prob_anemia >= THRESHOLD_ANEMIA else NORMAL_NAME
    return {
        "mode": "full_pipeline",
        "image_path": str(img_path),
        "prob_anemia": prob_anemia,
        "threshold_anemia": THRESHOLD_ANEMIA,
        "prediction": pred_label,
        "detection": det_info,
        "quality": quality,
    }

print("Función predict_single_image lista.")


## 21. Qué debe reportarse en la tesis

Mínimo defendible:

1. Fuente del dataset, licencia, número de imágenes y splits.
2. Advertencia de que las etiquetas son clínicas del dataset y no sustituyen hemoglobina medida, salvo que se cuente con ese dato.
3. Auditoría de duplicados y fuga de datos.
4. Detector anatómico de una clase: `conjuntiva`.
5. Clasificador entrenado con crops de conjuntiva.
6. Comparación de arquitecturas.
7. Umbral/calibración definidos con validación.
8. Test intocable.
9. Métricas con IC 95%.
10. Análisis de errores: falsos positivos, falsos negativos, no detecciones y casos de incertidumbre.

Conclusión aceptable:

> Se desarrolló un sistema de apoyo al tamizaje de anemia basado en imágenes de conjuntiva palpebral inferior. El sistema requiere validación clínica externa antes de considerarse herramienta diagnóstica.
